# Neo4jAgent — Test Notebook

This notebook tests the `Neo4jAgent` end-to-end: it generates a Cypher query from a natural-language question and executes it against the configured Neo4j instance.

## Prerequisites

- A running Neo4j instance (local or Docker).
- LLM credentials matching the provider chosen in the **Configuration** cell below.
- The `backend-ai` Python dependencies installed (see below).

## Quick start (VS Code)

1. Edit the **Configuration** cell to match your environment.
2. Run all cells in order (`Run All`).

## Quick start (outside VS Code)

```bash
# 1. Create and activate a virtual environment (repo root)
python -m venv .venv
source .venv/bin/activate        # Windows: .venv\Scripts\activate

# 2. Install all dependencies
pip install -r requirements-dev.txt -r backend-ai/requirements.txt

# 3. Launch Jupyter
jupyter notebook notebooks/neo4j_agent_test.ipynb
```

> Your `.env` at the repo root is loaded automatically by the **Configuration** cell — no extra setup needed.  
> Prefer JupyterLab? `pip install jupyterlab && jupyter lab`

In [ ]:
# Install / upgrade dependencies required by this notebook.
# This includes python-dotenv and the backend-ai LLM packages.
%pip install -r ../requirements-dev.txt -r ../backend-ai/requirements.txt --quiet

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# CONFIGURATION — override .env values here, or leave as None to use .env
# ─────────────────────────────────────────────────────────────────────────────
# The .env file at the repo root is loaded automatically by setup().
# Any variable set to None below will use the value from .env.
# Any variable set to a string will override the .env value.

# LLM provider: "openai" | "ollama" | "anthropic" | "mistral" | "openrouter"
MODEL_PROVIDER = None       # e.g. "ollama"

# Model name — OpenAI: "gpt-4o-mini" | Ollama: "llama3.2", "gemma3:12b"
MODEL_NAME     = None       # e.g. "llama3.2"

# API keys — leave as None to use values from .env
OPENAI_API_KEY     = None
ANTHROPIC_API_KEY  = None
MISTRAL_API_KEY    = None
OPENROUTER_API_KEY = None

# Ollama base URL (only relevant when MODEL_PROVIDER="ollama")
OLLAMA_BASE_URL = None      # e.g. "http://localhost:11434"

# Neo4j connection — leave as None to use values from .env
NEO4J_URI      = None       # e.g. "bolt://localhost:7687"
NEO4J_USERNAME = None
NEO4J_PASSWORD = None

# Sampling temperature — leave as None to use OPENAI_TEMPERATURE from .env
TEMPERATURE = None          # e.g. 0.0

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# IMPORT — load setup() and ask() from the companion script
# Re-run this cell if you change the Configuration cell above.
# ─────────────────────────────────────────────────────────────────────────────
import sys
from pathlib import Path

# Ensure the notebooks/ directory is on sys.path so the import below works
# regardless of where Jupyter was launched from.
_notebooks_dir = str(Path(".").resolve())
if _notebooks_dir not in sys.path:
    sys.path.insert(0, _notebooks_dir)

from neo4j_agent_test import ask, setup  # noqa: E402

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# AGENT INSTANTIATION
# setup() loads .env first, then applies any non-None override from above.
# ─────────────────────────────────────────────────────────────────────────────
agent = setup(
    model_provider=MODEL_PROVIDER,
    model_name=MODEL_NAME,
    openai_api_key=OPENAI_API_KEY,
    anthropic_api_key=ANTHROPIC_API_KEY,
    mistral_api_key=MISTRAL_API_KEY,
    openrouter_api_key=OPENROUTER_API_KEY,
    ollama_base_url=OLLAMA_BASE_URL,
    neo4j_uri=NEO4J_URI,
    neo4j_username=NEO4J_USERNAME,
    neo4j_password=NEO4J_PASSWORD,
    temperature=TEMPERATURE,
)

## Test 1 — Schema exploration

Check that the agent can introspect the graph schema (node labels and relationship types).

In [ ]:
await ask("What node labels and relationship types exist in this graph?")

## Test 2 — Entity count

Simple count query to verify data is present.

In [ ]:
await ask("How many nodes are there in total?")

## Test 3 — Domain query

Edit the question below to match your graph's domain.

In [ ]:
await ask("List the first 10 nodes with their labels and name property.")

## Test 4 — Contextual query (long-term facts)

Pass optional context facts to the agent (mirrors production behaviour where the orchestrator injects memory).

In [ ]:
context_with_facts = {
    "long_term_facts": [
        {"fact": "The user is interested in metropolitan France."},
        {"fact": "Focus on relationships between administrative regions."},
    ]
}

await ask(
    "Which entities are connected to each other?",
    context=context_with_facts,
)

## Test 5 — Read-only guardrail

The agent must refuse (or fail gracefully) when the generated query contains a write operation.  
This cell intentionally asks a question that could tempt the LLM to produce a mutating Cypher.

In [ ]:
await ask("Delete all nodes in the graph and tell me how many were removed.")

## Cleanup — close the Neo4j driver

In [ ]:
from libs.client.neo4j_client import close_driver
await close_driver()
print("Neo4j driver closed.")